# 02 — NAM Training & Shape Function Analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
import json

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

from src.data.split import load_splits
from src.models.nam_trainer import NAMTrainer
from src.visualization.shape_functions import (
    plot_shape_functions_grid, get_feature_importance_from_nam
)

## 1. Load Data

In [ ]:
splits = load_splits(config['data']['processed_dir'])
X_train = splits['X_train'].values
X_cal = splits['X_cal'].values
X_test = splits['X_test'].values
y_train = splits['y_train']
y_cal = splits['y_cal']
y_test = splits['y_test']
feature_names = splits['feature_names']

print(f'Train: {X_train.shape}, Cal: {X_cal.shape}, Test: {X_test.shape}')

## 2. Train NAM (with default config or search)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = NAMTrainer(device=device)

# Use default config for quick run; set skip_search=False for full search
nam_config = config['nam']
print(f'Training with config: {nam_config}')

In [ ]:
model, history = trainer.train_model(
    X_train, y_train, X_cal, y_cal, nam_config, verbose=True
)
print(f'\nBest validation AUC: {history["best_val_auc"]:.4f}')

## 3. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'])
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(history['val_auc'])
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Validation AUC')
plt.tight_layout()
plt.show()

## 4. Shape Functions

In [ ]:
# Feature importance
importance = get_feature_importance_from_nam(model, X_train)
top_idx = np.argsort(importance)[::-1]

print('Feature importance (variance of shape function):')
for i in top_idx:
    print(f'  {feature_names[i]:>12s}: {importance[i]:.6f}')

In [ ]:
# Plot top 6 shape functions
top_6 = top_idx[:6].tolist()
fig = plot_shape_functions_grid(model, top_6, X_train, feature_names, ncols=3)
plt.show()

In [ ]:
# Plot ALL shape functions
all_idx = list(range(len(feature_names)))
fig = plot_shape_functions_grid(model, all_idx, X_train, feature_names, ncols=4)
plt.show()

## 5. Test Set Evaluation

In [ ]:
from src.evaluation.metrics import compute_all_metrics

model.eval()
with torch.no_grad():
    logits, _ = model(torch.FloatTensor(X_test))
    y_prob = torch.sigmoid(logits).numpy().ravel()

metrics = compute_all_metrics(y_test, y_prob)
for k, v in metrics.items():
    print(f'{k:>15s}: {v:.4f}')